In [175]:
from pathlib import Path
import os
import tiktoken
import numpy as np
from tqdm import tqdm
from uuid import uuid4

from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.docstore.document import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_ollama.chat_models import ChatOllama
from langchain_pinecone import PineconeVectorStore
from langchain_classic.chains.conversation.memory import ConversationBufferWindowMemory
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_classic import hub
from langchain_classic.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder

from pinecone import Pinecone, ServerlessSpec

# Agents
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_classic.agents import AgentExecutor, Tool, create_tool_calling_agent
from langchain_classic.agents.react.agent import create_react_agent

In [69]:
PINECONE_API_KEY = Path("pinecone.tok").read_text().strip()
TAVILY_API_KEY = Path("tavily.tok").read_text().strip()

In [3]:
loader = CSVLoader(
    file_path="tedx.csv",
    encoding="utf-8",
    source_column="transcript",
    metadata_columns=["main_speaker", "name", "speaker_occupation", "title", "url", "description"]
)
data = loader.load()
len(data)

2467

In [4]:
tiktoken.encoding_name_for_model("gpt-3.5-turbo-0125")

'cl100k_base'

In [7]:
# embedding model picker
def cosine_similarity(query_emb, document_emb):
    dot_product = np.dot(query_emb, document_emb)
    query_norm = np.linalg.norm(query_emb)
    document_norm = np.linalg.norm(document_emb)
    return dot_product / (query_norm * document_norm)

def embed_and_compute_similarity(query, document, model_name):
    embeddding = OllamaEmbeddings(model=model_name)
    query_emb = embeddding.embed_query(query)
    document_emb = embeddding.embed_query(document)
    cos = cosine_similarity(query_emb, document_emb)
    print(f"Query Length: {len(query_emb)}")
    print(f"Document Length: {len(document_emb)}")
    print(f"Cosine Similarity: {cos}")

question = "What is the topic of the TEDx talk from Al Gore?"
document = "'Averting the climate crisis' is the topic"
embed_and_compute_similarity(question, document, "nomic-embed-text")


Query Length: 768
Document Length: 768
Cosine Similarity: 0.6579536751253554


In [8]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-5",
    chunk_size=512,
    chunk_overlap=20,
    separators=["\n\n", "\n", " ", ""]
)
splits = text_splitter.split_documents(data[:100])
len(splits)

717

In [9]:
index_name = "langchain-pinecone-test"
pc = Pinecone(api_key=PINECONE_API_KEY)

In [13]:
existing_indices = [index_info.name for index_info in pc.list_indexes()]
if index_name not in existing_indices:
    idx_model = pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(idx_model)
else:
    print(f"Index '{index_name}' already exists.")
# index = pc.index(index_name)

Index 'langchain-pinecone-test' already exists.


In [ ]:
index = pc.Index(index_name)
index.describe_index_stats()

{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

In [142]:
metaEmbededSplits = []
for split in splits:
    content = f"""
Title: {split.metadata['title']}
Main speaker: {split.metadata['main_speaker']}
Speaker occupation: {split.metadata['speaker_occupation']}
Description: {split.metadata['description']}

Transcript: {split.page_content}
"""
    metaEmbededSplits.append(Document(page_content=content, metadata=split.metadata))
embed = OllamaEmbeddings(model="nomic-embed-text")
vector_store = PineconeVectorStore(index=index, embedding=embed, namespace="main")
vector_store.add_documents(metaEmbededSplits)

['ba3aed2e-9504-4da9-bb3b-2846857e3f23',
 'a2b37d0a-d3d1-4e59-bc9b-8f47c361edb1',
 '0c137f00-b495-403f-82f6-393363787e9e',
 'f3a150aa-7398-4f17-8cd9-324855b284a5',
 '66118819-3d69-4f65-a83f-9f6dbba356e8',
 '08ad2189-a7ba-4a78-889d-f270773a157f',
 '77a2108c-95cc-4421-95d4-65aed7b84f63',
 'c0fa34d9-054c-4dbe-a341-93b23d080f0f',
 '5ca3f31d-88ec-40fb-8549-63945d7fd6ab',
 'e33ddc08-d70e-4bac-94ba-f10c65915c06',
 '8c9cca2e-feb6-48b3-b73e-cc0a6ff09ddc',
 '07dd4d51-049b-4be4-9f54-4014f456dd58',
 'e0b87062-3be2-4a9e-a845-4812b03a7543',
 '4195b5d8-e719-4fc5-8b4d-340ca1694299',
 'eb7782d0-c224-408b-b43a-5dba93f762ec',
 '4d9bd0bf-f0e2-4b76-82e6-0f1ea0d337c9',
 '39266bd9-4631-46f5-9125-1b39fb83c3bb',
 'c8c18e6c-cd70-4059-85d9-45abc2a4e9c2',
 '2a2bd154-0f72-4ea9-8130-dfd232e2c399',
 'dc7e8147-831f-4fdc-ae08-edea5743ba87',
 'ba225255-0da2-40ae-bd19-dbee4fccddf1',
 '61f133dd-b4eb-4a7f-a7e8-9c181320ed7d',
 '594914be-18e9-4936-8816-5504ae3fbb41',
 '13b446a6-8160-4aa8-b491-af46a7439e6c',
 'ba4ad503-94bc-

In [165]:
query = "Ken Robinson education creativity TED talk"
similarity = vector_store.similarity_search_with_score(query, k=4)
for i in range(len(similarity)):
    print(f"------Result {i+1}------")
    print(f"Score: {similarity[i][1]}")
    print(f"Main Speaker: {similarity[i][0].metadata['main_speaker']}")
    print()

------Result 1------
Score: 0.784969
Main Speaker: Ken Robinson

------Result 2------
Score: 0.774237394
Main Speaker: Ken Robinson

------Result 3------
Score: 0.767419577
Main Speaker: Ken Robinson

------Result 4------
Score: 0.764881134
Main Speaker: Ken Robinson



In [33]:
doc = splits[0]
embed.embed_documents([doc.page_content])

[[-0.021007057,
  0.06722456,
  -0.13196194,
  -0.037186842,
  0.060375955,
  0.04112448,
  0.03324612,
  -0.024666786,
  0.03335859,
  -0.05003534,
  -0.025831869,
  0.0053194202,
  0.064741865,
  0.064138345,
  0.04045017,
  -0.037094507,
  0.01384517,
  -0.06929128,
  -0.12042007,
  0.064486824,
  -0.025616072,
  -0.017483301,
  -0.017734727,
  0.018347383,
  0.0635995,
  0.031421658,
  -0.08692708,
  0.046129506,
  -0.06715528,
  -0.059085578,
  0.057181288,
  -0.085592754,
  -0.032680247,
  -0.0153383985,
  -0.100600906,
  -0.007367178,
  0.010773669,
  0.037760776,
  -0.04531002,
  -0.016905652,
  0.10256069,
  0.00047932376,
  -0.008010313,
  -0.07695862,
  0.020556329,
  -0.037872102,
  0.011490977,
  -0.06791318,
  0.027666507,
  -0.041115087,
  -0.00981349,
  0.032107834,
  0.03821238,
  0.05406377,
  0.04412769,
  0.039952274,
  0.02668734,
  0.004257819,
  0.018297706,
  0.015630946,
  0.08969452,
  -0.004701016,
  -0.035365578,
  0.057347562,
  -0.0014873226,
  -0.03618842

In [42]:
# add documents using upsert
def chunked_metadata_embeddings(documents):
    chucked_metadata = []
    chunked_text = text_splitter.split_documents(documents)
    for index, text in enumerate(tqdm(chunked_text)):
        payload = {
            "metadata": {
                "source": text.metadata['source'],
                "row": text.metadata['row'],
                "chunk_num": index,
                "main_speaker": text.metadata['main_speaker'],
                "name": text.metadata['name'],
                "speaker_occupation": text.metadata['speaker_occupation'],
                "title": text.metadata['title'],
                "url": text.metadata['url'],
                "description": text.metadata['description']
            },
            "id": str(uuid4()),
            "values": embed.embed_documents([text.page_content])[0]
        }
        chucked_metadata.append(payload)
    return chucked_metadata

def batch_upsert(split, namespace, batch_size):
    print(f"Split Length: {len(split)}")
    for i in range(0, len(split), batch_size):
        batch = split[i:i + batch_size]
        index.upsert(vectors=batch, namespace=namespace)

In [43]:
split_one = chunked_metadata_embeddings(data[:50])
len(split_one)

100%|██████████| 389/389 [01:16<00:00,  5.08it/s]


389

In [44]:
split_two = chunked_metadata_embeddings(data[50:100])
len(split_two)

100%|██████████| 328/328 [01:00<00:00,  5.44it/s]


328

In [45]:
batch_upsert(split_one, namespace="first_split", batch_size=10)

Split Length: 389


In [46]:
index.describe_index_stats()

{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'first_split': {'vector_count': 389},
                'main': {'vector_count': 717}},
 'total_vector_count': 1106,
 'vector_type': 'dense'}

In [47]:
batch_upsert(split_two, namespace="last_split", batch_size=10)

Split Length: 328


In [52]:
split_one_speakers = set()
split_two_speakers = set()
for split in split_one:
    split_one_speakers.add(split["metadata"]["main_speaker"])
for split in split_two:
    split_two_speakers.add(split["metadata"]["main_speaker"])

print("Split One Speakers:", split_one_speakers)
print("Split Two Speakers:", split_two_speakers)

Split One Speakers: {'Dean Ornish', 'Julia Sweeney', 'Majora Carter', 'Dan Gilbert', 'Bono', 'Richard Dawkins', 'Michael Shermer', 'Rives', 'Tony Robbins', 'Steven Levitt', 'Burt Rutan', 'Mena Trott', 'Al Gore', 'Peter Gabriel', 'Dan Dennett', 'Jeff Han', 'Jacqueline Novogratz', 'Eve Ensler', 'Ross Lovegrove', 'Jennifer Lin', 'Ken Robinson', 'Aubrey de Grey', 'Nicholas Negroponte', 'Malcolm Gladwell', 'Eva Vertes', 'Ze Frank', 'Peter Donnelly', 'David Pogue', 'Richard Baraniuk', 'Helen Fisher', 'Larry Brilliant', 'David Deutsch', 'Amy Smith', 'Robert Fischell', 'Kevin Kelly', 'Richard St. John', 'Joshua Prince-Ramus', 'Ray Kurzweil', 'Cameron Sinclair', 'Ben Saunders', 'Barry Schwartz', 'Iqbal Quadir', 'Sirena Huang', 'Rick Warren', 'Ashraf Ghani', 'Jehane Noujaim', 'Sasa Vucinic', 'Edward Burtynsky', 'Jimmy Wales', 'Hans Rosling'}
Split Two Speakers: {'Phil Borges', 'Janine Benyus', 'Carl Honoré', 'Saul Griffith', 'Caroline Lavelle', 'Craig Venter', 'Susan Savage-Rumbaugh', 'Dean Kame

In [55]:
query_one = "What does Julia Sweeney do?"
query_two = "Who is Saul Griffith?"

query_vector_one = embed.embed_query(query_one)
query_vector_two = embed.embed_query(query_two)
print(len(query_vector_one), len(query_vector_two))

768 768


In [56]:
index.query(vector=query_vector_one, top_k=2, include_metadata=True, namespace="first_split")

{'matches': [{'id': '75a344b0-8fa9-4bee-a2e4-16390c00094f',
              'metadata': {'chunk_num': 42.0,
                           'description': 'Tony Robbins discusses the '
                                          '"invisible forces" that motivate '
                                          "everyone's actions -- and "
                                          'high-fives Al Gore in the front '
                                          'row.',
                           'main_speaker': 'Tony Robbins',
                           'name': 'Tony Robbins: Why we do what we do',
                           'row': 5.0,
                           'source': "Thank you. I have to tell you I'm both "
                                     'challenged and excited. My excitement '
                                     'is: I get a chance to give something '
                                     'back. My challenge is: the shortest '
                                     'seminar I usually do is 50 

In [58]:
index.query(vector=query_vector_two, top_k=2, include_metadata=True, namespace="last_split")

{'matches': [{'id': 'c9544376-c15f-4d5b-8dbb-d346c068aefe',
              'metadata': {'chunk_num': 109.0,
                           'description': 'As E.O. Wilson accepts his 2007 TED '
                                          'Prize, he makes a plea on behalf of '
                                          'all creatures that we learn more '
                                          'about our biosphere -- and build a '
                                          'networked encyclopedia of all the '
                                          "world's knowledge about life.",
                           'main_speaker': 'E.O. Wilson',
                           'name': 'E.O. Wilson: My wish: Build the '
                                   'Encyclopedia of Life',
                           'row': 63.0,
                           'source': 'I have all my life wondered what '
                                     '"mind-boggling" meant. After two days '
                                     'her

In [158]:
llm = ChatOllama(
    model="qwen3:1.7b",
    validate_model_on_init=True,
    temperature=0.0,
    num_predict=512
)

conversational_memory = ConversationBufferWindowMemory(
    memory_key="chat_history",
    k=5,
    return_messages=True
)

qa_db = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever()
)

In [63]:
prompt = hub.pull("hwchase17/react")
prompt

/var/folders/dc/kw3ll4h16l593j6myd2kj7vr0000gp/T/ipykernel_75666/374980370.py:1: LangChainDeprecationWarning: langchain_classic.hub.pull is deprecated. Use the LangSmith SDK instead.
  prompt = hub.pull("hwchase17/react")


ValueError: Pulling a public prompt by owner/name is disabled by default because prompts may contain untrusted serialized LangChain objects. If you trust this prompt, set `dangerously_pull_public_prompt=True` to acknowledge the risk.

In [ ]:
template= '''
        Answer the following questions as best you can. You have access to the following tools:

        {tools}

        Use the following format:

        Question: the input question you must answer
        Thought: you should always think about what to do
        Action: the action to take, should be one of [{tool_names}]. Always look first in Pinecone Document Store
        Action Input: a plain-text query only. Do not pass dictionaries, JSON, or metadata parameters.
        Observation: the result of the action
        ... (this Thought/Action/Action Input/Observation can repeat 2 times)
        Thought: I now know the final answer
        Final Answer: the final answer to the original input question

        Begin!

        Question: {input}
        Thought:{agent_scratchpad}
'''

template = """
Answer the question using the available tools.

Tools:
{tools}

You must use exactly one of these two formats.

To call a tool:

Thought: explain the next step
Action: one of [{tool_names}]
Action Input: a plain-text query

When an Observation contains the answer, stop immediately and use:

Thought: I now know the final answer
Final Answer: the answer

Rules:
- Never use "Answer:" by itself.
- Never put the answer in an Action field.
- Never write Markdown around Action, Action Input, or Final Answer.
- Never call a tool again when its Observation contains the answer.
- Search Pinecone first.
- If Pinecone does not contain the answer, search Tavily once.

Question: {input}
Thought:{agent_scratchpad}
"""

prompt = PromptTemplate.from_template(template)

In [176]:
# Set up tools and agent
tavily = TavilySearchResults(max_results=3, tavily_api_key=TAVILY_API_KEY)
tavily.run("Lord of the Mysteries")

[{'title': 'Lord of Mysteries - Wikipedia',
  'url': 'https://en.wikipedia.org/wiki/Lord_of_Mysteries',
  'content': 'Lord of Mysteries (Chinese: 诡秘之主; pinyin: Guǐ Mì Zhī Zhǔ) is a Chinese web novel written by Yuan Ye/Cuttlefish That Loves Diving. Serialized on Qidian from 2018 to 2020, the story blends steampunk, Lovecraftian horror, and Xuanhuan elements in a Victorian era-inspired world.\n\n## Plot\n\n[edit] [...] 2018 Chinese web novel\n\nLord of Mysteries\n\n| Volume 1 print cover |\n| Author | Cuttlefish That Loves Diving |\n| Original title | 诡秘之主 |\n| Translator | Webnovel") [zh] |\n| Illustrator | OSSS |\n| Country | China |\n| Language | Mandarin Chinese |\n| Genre |  Dark fantasy  Lovecraftian horror  Steampunk  Xuanhuan |\n| Publisher |  Qidian (web serial)  China Literature Ltd.") [zh] (print)  Yen Press (English print) |\n| Published |  April 1, 2018 (2018-04-01) – May 1, 2020 (2020-05-01) (web serial)  2020 (2020) – present (print) |\n| Published in English | July 29, 20

In [172]:
tools = [
    Tool(
        name="Pinecone Document Store",
        func=qa_db.run,
        description=(
            "Use it to lookup information from the Pinecone Document Store. "
            "Pass a concise plain-text query that preserves names "
            "and requested metadata fields such as title, speaker, and event."
        )
    ),
    Tool(
        name="Tavily",
        func=tavily.run,
        description="Use this to lookup information from Tavily"
    )
]

agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    tools=tools,
    agent=agent,
    handle_parsing_errors=False,
    verbose=True,
    memory=conversational_memory,
    # max_iterations=2,
    # max_execution_time=60
)

In [163]:
metadata = "name"
value = "The surprising science of happiness"
metaToGet = "main_speaker"
res = set()
for split in splits:
    if value.lower() in split.metadata[metadata].lower():
        res.add(split.metadata[metaToGet])

res

{'Dan Gilbert'}

In [153]:
input = {
    "input": "Can you give me one title of a TED talk of Dan Gilbert as main speaker?. Please look in the pinecone document store metadata as it has the title based on the transcripts"
}

In [173]:
agent_executor.invoke(input)



> Entering new AgentExecutor chain...
Thought: I will use the Pinecone Document Store to look up the title of a TED talk by Dan Gilbert based on metadata fields like title.  
Action: Pinecone Document Store  
Action Input: "title"  
Vik Muniz is an artist, Nora York is a singer and performance artist, and Robert Wright is a journalist and philosopher.Thought: Since the Pinecone Document Store did not provide the desired title, I will use the Tavily tool to search for the specific TED talk title by Dan Gilbert.  
Action: Tavily  
Action Input: "Dan Gilbert TED talk title"  
[{'title': 'Daniel Gilbert | Speaker | TED', 'url': 'https://www.ted.com/speakers/dan_gilbert', 'content': 'Daniel Gilbert\n\n# Daniel Gilbert\n\nHarvard psychology professor\n\n## Why you should listen\n\n## Daniel Gilbert’s TED Talks\n\nThe surprising science of happiness\n\nDaniel Gilbert\n\nThe surprising science of happiness\n\nWhy we make bad decisions\n\nDaniel Gilbert\n\nWhy we make bad decisions\n\nThe psy

ValueError: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: `The title of the TED talk by Dan Gilbert that is highlighted in the Pinecone Document Store's metadata is **"The psychology of your future self"**. This title is explicitly mentioned in the results, aligning with his research on the "end of history illusion" and the concept of human transformation. 

**Answer:**  
**The psychology of your future self**.`
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

In [167]:
qa_db.run("Kevin Robinson TED talk title")

"I don't know the title of Kevin Robinson's TED talk based on the provided context. The context discusses Seth Godin's talks, Nora York's performance, and other topics, but there is no mention of Kevin Robinson's work."

In [137]:
inp = {
    "input": "Did you find the previous title 'The Case for Optimism on Climate Change' in the Pinecone Document Store?"
}

agent_executor.invoke(inp)



> Entering new AgentExecutor chain...
Invalid Format: Missing 'Action:' after 'Thought:'Action: Pinecone Document Store
Action Input: "The Case for Optimism on Climate Change"The provided text discusses the "International System for Total Early Disease Detection" (INSTEDD) and its focus on early detection of health threats, but it does not explicitly address climate change or optimism regarding climate change. The content emphasizes global health, early detection, and collaborative efforts to prevent disasters, rather than climate-related issues. Since the question pertains to climate change and the context does not cover this topic, the answer is that the provided material does not discuss climate change or optimism on it.Thought: The Pinecone Document Store query for "The Case for Optimism on Climate Change" did not return any relevant results, as the provided text focuses on INSTEDD and health threats, not climate change. Thus, the title is not present in the documented content.  

{'input': "Did you find the previous title 'The Case for Optimism on Climate Change' in the Pinecone Document Store?",
 'chat_history': [HumanMessage(content='Can you give me one title of a TED talk of Dan Gilbert as main speaker?.         Please look in the pinecone document store metadata as it has the title based on the transcripts', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The title of Dan Gilbert\'s TED talk is **"The Encyclopedia of Life."**', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'output': 'No, the title "The Case for Optimism on Climate Change" was not found in the Pinecone Document Store. The provided text discusses unrelated topics like early disease detection, not climate change.'}

In [187]:
agent_executor.memory.clear()

In [181]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Search the Pinecone document store first. If it does not contain the answer, search Tavily. When a tool result contains the answer, answer the user directly instead of calling another tool."),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad")
])

tools = [tools[0], tavily]

agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    memory=conversational_memory,
    max_iterations=10
)

In [184]:
agent_executor.invoke(input)



> Entering new AgentExecutor chain...

Invoking: `Pinecone Document Store` with `{'query': 'Dan Gilbert TED talk title'}`


The provided context does not mention Dan Gilbert's TED talk title. It focuses on Seth Godin's talks about idea diffusion and Nora York's performance of "What I Want." Therefore, I cannot determine the title of Dan Gilbert's TED talk from the given information.
Invoking: `tavily_search_results_json` with `{'query': 'Dan Gilbert TED talk title'}`


[{'title': 'Daniel Gilbert | Speaker | TED', 'url': 'https://www.ted.com/speakers/dan_gilbert', 'content': 'Daniel Gilbert\n\n# Daniel Gilbert\n\nHarvard psychology professor\n\n## Why you should listen\n\n## Daniel Gilbert’s TED Talks\n\nThe surprising science of happiness\n\nDaniel Gilbert\n\nThe surprising science of happiness\n\nWhy we make bad decisions\n\nDaniel Gilbert\n\nWhy we make bad decisions\n\nThe psychology of your future self\n\nDaniel Gilbert\n\nThe psychology of your future self\n\nDan Gilbert on the 

{'input': 'Can you give me one title of a TED talk of Dan Gilbert as main speaker?. Please look in the pinecone document store metadata as it has the title based on the transcripts',
 'chat_history': [],
 'output': 'The title of Dan Gilbert\'s TED talk is **"The Surprising Science of Happiness"**. This talk, first delivered in 2004, explores how humans misjudge what brings them happiness and highlights the science behind emotional decision-making.'}

In [186]:
inp = {
    "input": "Did you find the previous title 'The Surprising Science of Happiness' in the Pinecone Document Store?"
}

agent_executor.invoke(inp)



> Entering new AgentExecutor chain...

Invoking: `Pinecone Document Store` with `The Surprising Science of Happiness`


The provided context does not mention "The Surprising Science of Happiness" or any related topics. Therefore, I cannot answer the question based on the given information.The title **"The Surprising Science of Happiness"** is not listed in the Pinecone Document Store based on the provided context. However, if you have another title in mind (e.g., related to Dan Gilbert or other topics), I can check that for you!

> Finished chain.


{'input': "Did you find the previous title 'The Surprising Science of Happiness' in the Pinecone Document Store?",
 'chat_history': [HumanMessage(content='Can you give me one title of a TED talk of Dan Gilbert as main speaker?. Please look in the pinecone document store metadata as it has the title based on the transcripts', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The title of Dan Gilbert\'s TED talk is **"The Surprising Science of Happiness"**. This talk, first delivered in 2004, explores how humans misjudge what brings them happiness and highlights the science behind emotional decision-making.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="Did you find the previous title 'The Surprising Science of Happiness' in the Pinecone Document Store?", additional_kwargs={}, response_metadata={}),
  AIMessage(content='', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'out

In [188]:
inp = {
    "input": "What is the main idea in Ken Robinson education creativity TED talk?"
}

agent_executor.invoke(inp)



> Entering new AgentExecutor chain...

Invoking: `Pinecone Document Store` with `{'query': 'Ken Robinson education creativity TED talk main idea'}`


Ken Robinson's TED talk on education and creativity argues that the current education system often stifles creativity by prioritizing standardization, conformity, and academic ability over nurturing diverse talents and innovative thinking. He challenges the notion that creativity is solely reserved for "gifted" individuals, emphasizing that all children possess creative potential. Robinson highlights how schools often suppress natural talents by focusing on grades and rigid curricula, leading to a culture of "academic inflation" where degrees become more prestigious than practical skills. He advocates for rethinking education to value creativity, diversity of thought, and real-world application, stressing that creativity is as vital as literacy and must be cultivated rather than undermined. The main idea is that education systems need t

{'input': 'What is the main idea in Ken Robinson education creativity TED talk?',
 'chat_history': [],
 'output': 'Ken Robinson\'s TED talk on education and creativity argues that the modern education system often suppresses creativity by prioritizing standardization, conformity, and academic grades over nurturing diverse talents and innovation. He challenges the idea that creativity is limited to "gifted" individuals, emphasizing that all children have creative potential. Robinson critiques rigid curricula and the pressure to achieve high grades, highlighting how this stifles natural talents. He advocates for education systems that value diversity, real-world application, and interdisciplinary thinking, stressing that creativity is essential for future innovation and should be cultivated rather than undermined. The core message is the need to transform education to foster creativity, recognize individual differences, and prepare students for a world driven by innovation.'}

In [189]:
inp = {
    "input": "What was the title of the previous document found in the Pinecone Document Store?"
}

agent_executor.invoke(inp)



> Entering new AgentExecutor chain...

Invoking: `Pinecone Document Store` with `title`


The titles are:  
1. *Art with wire, sugar, chocolate and string*  
2. *If I controlled the Internet*  
3. *Four American characters*  
4. *Progress is not a zero-sum game*The title of the previous document found in the Pinecone Document Store is **"If I controlled the Internet"**.

> Finished chain.


{'input': 'What was the title of the previous document found in the Pinecone Document Store?',
 'chat_history': [HumanMessage(content='What is the main idea in Ken Robinson education creativity TED talk?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Ken Robinson\'s TED talk on education and creativity argues that the modern education system often suppresses creativity by prioritizing standardization, conformity, and academic grades over nurturing diverse talents and innovation. He challenges the idea that creativity is limited to "gifted" individuals, emphasizing that all children have creative potential. Robinson critiques rigid curricula and the pressure to achieve high grades, highlighting how this stifles natural talents. He advocates for education systems that value diversity, real-world application, and interdisciplinary thinking, stressing that creativity is essential for future innovation and should be cultivated rather than undermined. The core message is

In [192]:
# agent_executor.memory.load_memory_variables({})
conversational_memory.load_memory_variables({})

{'chat_history': [HumanMessage(content='What is the main idea in Ken Robinson education creativity TED talk?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Ken Robinson\'s TED talk on education and creativity argues that the modern education system often suppresses creativity by prioritizing standardization, conformity, and academic grades over nurturing diverse talents and innovation. He challenges the idea that creativity is limited to "gifted" individuals, emphasizing that all children have creative potential. Robinson critiques rigid curricula and the pressure to achieve high grades, highlighting how this stifles natural talents. He advocates for education systems that value diversity, real-world application, and interdisciplinary thinking, stressing that creativity is essential for future innovation and should be cultivated rather than undermined. The core message is the need to transform education to foster creativity, recognize individual differences, and pr